In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [2]:


train = np.load("./data/processed/TRAIN_30.npz", allow_pickle=True)
X_train = train["X"][::20]
y_train = train["y"][::20]

full = np.load("./data/processed/FULL_30.npz", allow_pickle=True)
X_full  = full["X"]
Delta_full = full["Delta"]


In [3]:
cols_20 = np.setdiff1d(
    np.arange(30),
    [2,3,5,7,12,18,21,24,27,28]
)
cols_12 = [0,2,4,5,6,11,12,15,17,23,24,29]

In [4]:
feature_sets = {
    "30": np.arange(30),
    "20": cols_20,
    "12": cols_12,
}

In [5]:
summary_rows = []
mean_rows = []
chi_rows = []

In [6]:
def build_model(base_model, calibrated=False):
    pipe = Pipeline([
        # ("scaler", StandardScaler()),
        ("clf", base_model)
    ])
    if not calibrated:
        return pipe

    return CalibratedClassifierCV(pipe)

In [7]:
def run_supervised_pipeline(model, X_train, y_train, X_full):
    model.fit(X_train, y_train)
    train_score = model.score(X_train, y_train)

    if hasattr(model, "predict_proba"):
        P_B = model.predict_proba(X_full)[:, 1]
    else:
        scores = model.decision_function(X_full)
        P_B = 1.0 / (1.0 + np.exp(-scores))

    return train_score, P_B

In [8]:
def stat_mean(x):
    return np.mean(x)



def stat_susceptibility(x):
    mu = np.mean(x)

    if abs(mu) < 1e-12:
        return 0.0

    return np.var(x, ddof=1)
    

In [9]:
def block_jackknife(data, stat_func, B):

    n = len(data)
    m = n // B  

    if m < 1:
        return None

    data = data[:m * B]

    blocks = data.reshape(B, m)

    theta_hat = stat_func(data)

    dats = np.array([
        stat_func(np.concatenate([blocks[:i], blocks[i+1:]]).ravel())
        for i in range(B)
    ])

    bias = (B - 1) * (np.mean(dats) - theta_hat)

    theta_jack = theta_hat - bias

    var = np.var(dats, ddof=1) * (B - 1)**2 / B
    se = np.sqrt(var)

    return theta_hat, theta_jack, se



In [10]:
# =============================================================================
# JACKKNIFE OVER Delta — returns FULL TABLE for every B
# =============================================================================

def jackknife_over_Delta(P_B, Delta, stat_func, B_range=range(2, 101)):
    df_input = pd.DataFrame({"Delta": Delta, "P_B": P_B})
    results = {}

    for d, group in df_input.groupby("Delta"):
        data = group["P_B"].values

        if len(data) < 2:
            continue

        rows = []
        for B in B_range:
            out = block_jackknife(data, stat_func, B)
            if out is None:
                continue
            theta_hat, theta_jack, se = out
            rows.append({
                "B":          B,
                "theta_hat":  theta_hat,   
                "theta_jack": theta_jack,  
                "se":         se,          
            })

        results[d] = pd.DataFrame(rows)
    return results

In [11]:
def summarise_jackknife(results_dict):

    Delta_vals, theta_arr, se_arr = [], [], []

    for d in sorted(results_dict.keys()):
        df = results_dict[d]

        if df.empty:
            continue

        df_valid = df.dropna(subset=["se"])

        if df_valid.empty:
            print(f"All SE are NaN for Delta={d}")
            continue

        row = df_valid.loc[df_valid["se"].idxmax()]

        Delta_vals.append(d)
        theta_arr.append(row["theta_jack"])
        se_arr.append(row["se"])

    return (
        np.array(Delta_vals),
        np.array(theta_arr),
        np.array(se_arr),
    )



def plot_se_vs_B(results_dict, stat_label, target_Deltas=None, ncols=5):
    ds = sorted(results_dict.keys())
    if target_Deltas is not None:
        ds = [k for k in ds if k in target_Deltas]

    n = len(ds)
    if n == 0:
        return

    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), squeeze=False)

    for idx, k in enumerate(ds):
        row, col = divmod(idx, ncols)
        ax = axes[row][col]
        df = results_dict[k]
        ax.plot(df["B"], df["se"], lw=1.5)
        ax.set_title(f"Delta = {k:.3f}")
        ax.set_xlabel("B (number of blocks)")
        ax.set_ylabel(f"SE of {stat_label}")
        ax.grid(True, alpha=0.3)

    for idx in range(n, nrows * ncols):
        row, col = divmod(idx, ncols)
        axes[row][col].set_visible(False)

    fig.suptitle(f"SE vs B — {stat_label} (should plateau)", y=1.02)
    plt.tight_layout()

In [12]:
from scipy.interpolate import interp1d
from scipy.optimize import brentq


def find_delta_crit(D, mean, mean_err):

    idx_close = np.argmin(np.abs(mean-0.5))
    d_close = D[idx_close]
    
    f = interp1d(
        D,
        mean - 0.5,
        kind="linear",
        bounds_error=False
    )

    idx = np.where(
        (mean[:-1] - 0.5) *
        (mean[1:]  - 0.5) <= 0
    )[0]

    if len(idx) == 0:
        return np.nan, np.nan, np.nan

    i = idx[0]

    dcrit = brentq(
        lambda x: f(x),
        D[i],
        D[i + 1]
    )

    slope = (
        mean[i + 1] - mean[i]
    ) / (
        D[i + 1] - D[i]
    )

    sigma_p = np.interp(
        dcrit,
        [D[i], D[i + 1]],
        [mean_err[i], mean_err[i + 1]]
    )
    if abs(slope) < 1e-12:
        return dcrit, np.nan, d_close
    
    sigma_d = sigma_p / abs(slope)

    return dcrit, sigma_d, d_close


In [13]:
def timed(func, *args, **kwargs):
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    return result, elapsed

In [14]:

# =============================================================================
# MAIN
# =============================================================================

models = {
    # "Logistic Regression": LogisticRegression(max_iter=2000),
    # "Logistic Regression C = 0.00175": LogisticRegression(C=0.00175),
    # "Decision Tree": DecisionTreeClassifier(random_state=0),
    # "Random Forest": RandomForestClassifier(n_estimators=200, random_state=0),
    # "Gradient Boosted Trees": GradientBoostingClassifier(random_state=0),
    # "kNN": KNeighborsClassifier(n_jobs=-1),
    # "kNN kd_tree": KNeighborsClassifier(algorithm="kd_tree", n_jobs=-1),
    # "kNN ball_tree": KNeighborsClassifier(algorithm="ball_tree",n_jobs=-1),
    # "kNN brute": KNeighborsClassifier(algorithm="brute",n_jobs=-1),
    "SVM (RBF)": SVC(kernel="rbf"),
    # "Neural Network": MLPClassifier(hidden_layer_sizes=(50,50), max_iter=500)
}
curves = {}

for feature_name, cols in feature_sets.items():

    X_train_fs = X_train[:, cols]
    X_full_fs  = X_full[:, cols]

    curves[feature_name] = {}

    print(
        f"\n{'='*70}"
        f"\nFEATURE SET: {feature_name}"
        f" ({len(cols)} features)"
        f"\n{'='*70}"
    )
            
    for name, base_model in models.items():
        print(f"\n{'='*50}\nModel: {name}\n{'='*50}")
        curves[feature_name][name] = {}
    
        for calibrated in [True, False]:
            label = "calibrated" if calibrated else "uncalibrated"
            model = build_model(base_model, calibrated=calibrated)
    
            (train_score, P_B), model_time = timed(
                    run_supervised_pipeline,
                    model,
                    X_train_fs,
                    y_train,
                    X_full_fs,
                )
    
    
            
           
            mean_results, mean_jack_time = timed(
                jackknife_over_Delta,
                P_B,
                Delta_full,
                stat_mean,
            )
            
            chi_results, chi_jack_time = timed(
                jackknife_over_Delta,
                P_B,
                Delta_full,
                stat_susceptibility,
            )
            
            print(f"  Delta values found: {len(mean_results)}")        
            # Summarise using plateau average (physically correct)
            (D_mean, mean_jack, mean_se), summary_mean_time = timed(
                summarise_jackknife,
                mean_results,
            )
            
            (D_chi,  chi_jack,  chi_se), summary_chi_time = timed(
                summarise_jackknife,
                chi_results,
            )
    
            (dcrit, dcrit_err, d_close), time_delta_crit = timed(
                find_delta_crit,
                D_mean,
                mean_jack,
                mean_se,
            )
            
            
            total_time = model_time + mean_jack_time + chi_jack_time + summary_mean_time + summary_chi_time + time_delta_crit
            print(f"{label:12s} | train_acc={train_score:.4f} | total_time={round(total_time, 3)}")
            # curves[name][label] = {
            #     "K":      K_mean,
            #     "mean":   mean_jack,
            #     "mean_se": mean_se,
            #     "chi":    chi_jack,
            #     "chi_se": chi_se,
            #     # Keep raw tables for diagnostics
            #     "_mean_results": mean_results,
            #     "_chi_results":  chi_results,
            # }
            curves[feature_name][name][label] = {
                "train_acc": train_score,
                "runtime_sec": total_time,
            
                "D": D_mean,
            
                "mean": mean_jack,
                "mean_se": mean_se,
            
                "chi": chi_jack,
                "chi_se": chi_se,
            
                "Delta_crit": dcrit,
                "Delta_crit_err": dcrit_err,
                "Delta_close": d_close,
    
                
                "_mean_results": mean_results,
                "_chi_results": chi_results,
            }
            summary_rows.append({
                "feature_set": feature_name,
                "n_features": len(cols),
                "model": name,
                "calibration": label,
                "train_acc": train_score,
                "Delta_crit": dcrit,
                "Delta_crit_err": dcrit_err,
                "Delta_close": d_close,
                "N_Delta": len(D_mean),
                 "runtime_model": round(model_time, 3),
                "runtime_jackknife_mean": round(mean_jack_time, 3),
                "runtime_jackknife_chi": round(chi_jack_time, 3),
                "summary_mean_time": round(summary_mean_time, 4),
                "summary_chi_time": round(summary_chi_time, 4),
                "runtime_delta_crit": round(time_delta_crit, 5),
                "runtime_total": round(total_time, 3),        
            })
    
            for d, m, err in zip(
                D_mean,
                mean_jack,
                mean_se):
    
                mean_rows.append({
                    "feature_set": feature_name,
                    "n_features": len(cols),
                    "model": name,
                    "calibration": label,
                    "Delta": d,
                    "mean": m,
                    "mean_err": err,
                })
            for d, c, err in zip(
            D_chi,



            chi_jack,
            chi_se):
    
                chi_rows.append({
                    "feature_set": feature_name,
                    "n_features": len(cols),
                    "model": name,
                    "calibration": label,
                    "Delta": d,
                    "chi": c,
                    "chi_err": err,
                })    
    



FEATURE SET: 30 (30 features)

Model: SVM (RBF)
  Delta values found: 11
calibrated   | train_acc=1.0000 | total_time=39.496
  Delta values found: 11
uncalibrated | train_acc=1.0000 | total_time=14.94

FEATURE SET: 20 (20 features)

Model: SVM (RBF)
  Delta values found: 11
calibrated   | train_acc=1.0000 | total_time=21.075
  Delta values found: 11
uncalibrated | train_acc=1.0000 | total_time=10.5

FEATURE SET: 12 (12 features)

Model: SVM (RBF)
  Delta values found: 11
calibrated   | train_acc=1.0000 | total_time=29.27
  Delta values found: 11
uncalibrated | train_acc=1.0000 | total_time=11.237


In [15]:
import os

os.makedirs("results", exist_ok=True)

pd.DataFrame(summary_rows).to_csv(
    "results/summarysvm.csv",
    index=False
)

pd.DataFrame(mean_rows).to_csv(
    "results/curve_meansvm.csv",
    index=False
)

pd.DataFrame(chi_rows).to_csv(
    "results/curve_chisvm.csv",
    index=False
)